# 原始数据问题复核与处理预演

本 Notebook 重算本次 **15 组重复冲突与 13 条字符告警**的复核建议，并对全部原始记录进行内存预演。它调用 `build_review()`，不调用命令行入口，不生成处理后数据，也不应用隔离或文本修改。

复核意见是辅助判断，不是权威替代标签。原始扫描报告和固定验证标签均不改动；两条保留的跨集合验证记录仍存在标签争议。


In [1]:
from pathlib import Path
from collections import Counter
import json
import sys

# 支持从仓库根目录或 notebooks 目录启动；不依赖个人机器绝对路径。
candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((p for p in candidates if
                  (p / "scripts" / "review_dataset_issues.py").is_file() and
                  (p / "data" / "reports" / "raw" / "validation_report.json").is_file()), None)
if repo_root is None:
    raise RuntimeError("请从本仓库或 notebooks 目录运行本 Notebook。")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from scripts.review_dataset_issues import build_review

# 只读取证据与原始数据，在内存返回建议和全量预演摘要。
decisions, summary = build_review(repo_root / "data" / "reports" / "raw" / "validation_report.json")
assert all(not d["action_applied"] for d in decisions)
assert summary["implementation_status"] == "not_applied"
assert summary["raw_data_modified"] is False
assert summary["original_validation_report_modified"] is False
print("复核加载完成；动作尚未应用，原始数据及校验报告未修改。")


复核加载完成；动作尚未应用，原始数据及校验报告未修改。


In [2]:
duplicate_action_types = {"quarantine_training_conflict_cluster", "quarantine_training_overlap"}
def record_keys(items):
    return {(r["split"], r["record_number"]) for item in items
            for r in item["action"].get("quarantine", [])}

duplicate_keys = record_keys([d for d in decisions if d["action"]["type"] in duplicate_action_types])
corrupted_keys = record_keys([d for d in decisions if d["action"]["type"] == "quarantine_corrupted_training_record"])
assert not duplicate_keys.intersection(corrupted_keys)
assert len(duplicate_keys) == 29 and len(corrupted_keys) == 2
assert summary["quarantine_count"] == len(duplicate_keys | corrupted_keys) == 31
assert summary["reviewed_issue_count"] == 28
assert summary["reviewed_issue_counts"] == {"WARN": 13, "REVIEW": 15}
assert summary["dry_run"]["expected_after_counts"] == {"train": 104969, "validation": 15000}

counts = {
    "reviewed_issue_counts": summary["reviewed_issue_counts"],
    "proposed_quarantine_duplicate_training_records": len(duplicate_keys),
    "proposed_quarantine_corrupted_training_records": len(corrupted_keys),
    "proposed_quarantine_total": summary["quarantine_count"],
    "proposed_cleaned_record_counts": summary["proposed_cleaned_record_counts"],
    "proposed_removed_control_characters": summary["proposed_removed_control_characters"],
    "actual_before_counts": summary["dry_run"]["actual_before_counts"],
    "expected_after_counts": summary["dry_run"]["expected_after_counts"],
    "action_applied": False,
}
print(json.dumps(counts, ensure_ascii=False, indent=2))


{
  "reviewed_issue_counts": {
    "WARN": 13,
    "REVIEW": 15
  },
  "proposed_quarantine_duplicate_training_records": 29,
  "proposed_quarantine_corrupted_training_records": 2,
  "proposed_quarantine_total": 31,
  "proposed_cleaned_record_counts": {
    "train": 10,
    "validation": 1
  },
  "proposed_removed_control_characters": 21,
  "actual_before_counts": {
    "train": 105000,
    "validation": 15000
  },
  "expected_after_counts": {
    "train": 104969,
    "validation": 15000
  },
  "action_applied": false
}


## 全量内存预演

以下统计把建议隔离与已复核控制字符删除应用到内存中的候选记录，检查新空文本、异常字符、重复和跨集合重叠；不写出候选 CSV。它覆盖本次明确规则，不等于全量语义去重或全部标签质量审计。


In [3]:
dry_run = summary["dry_run"]
check_names = [
    "candidate_empty_texts", "candidate_suspicious_character_records",
    "candidate_exact_duplicate_groups", "candidate_normalized_duplicate_groups",
    "candidate_cross_split_normalized_groups", "cleanup_collision_groups",
]
for name in check_names:
    assert dry_run[name] == 0, (name, dry_run[name])
assert dry_run["all_four_classes_remain_per_aspect_per_split"] is True
assert dry_run["processed_files_written"] is False
assert all(not decision["label_changes"] for decision in decisions)
checks = {name: dry_run[name] for name in check_names}
checks["all_four_classes_remain_per_aspect_per_split"] = dry_run["all_four_classes_remain_per_aspect_per_split"]
checks["processed_files_written"] = dry_run["processed_files_written"]
checks["labels_changed"] = False
print(json.dumps(checks, ensure_ascii=False, indent=2))


{
  "candidate_empty_texts": 0,
  "candidate_suspicious_character_records": 0,
  "candidate_exact_duplicate_groups": 0,
  "candidate_normalized_duplicate_groups": 0,
  "candidate_cross_split_normalized_groups": 0,
  "cleanup_collision_groups": 0,
  "all_four_classes_remain_per_aspect_per_split": true,
  "processed_files_written": false,
  "labels_changed": false
}


## 15 组重复与标签冲突

只展示复核决定中筛选的证据短引文，不输出完整原文。相同非空白内容的空白排版变体不能解释标签分歧；先隔离训练冲突簇，恢复训练样本前再按一致的 20 方面规范重标。跨集合问题保留固定验证记录及其原标签。


In [4]:
def references(records):
    return ", ".join(f"{r['split']}:{r['id']}" for r in records)

def show_quote(quote):
    # 保留汉字可读性；控制字符与换行按转义形式呈现。
    return json.dumps(quote, ensure_ascii=False).replace(chr(127), r"\u007f")

review_items = [d for d in decisions if d["original_severity"] == "REVIEW"]
for index, decision in enumerate(review_items, 1):
    print(f"{index:02d}. {decision['issue_id']} | {references(decision['affected_records'])}")
    print("   建议动作:", decision["action"]["type"])
    print("   建议隔离:", references(decision["action"].get("quarantine", [])))
    retained = decision["action"].get("preserve_fixed_validation", [])
    if retained:
        print("   保留固定验证:", references(retained), "（标签争议仍在）")
    print("   冲突方面:", ", ".join(decision["label_conflicts"]))
    for quote in decision["evidence_quotes"]:
        print("   证据:", show_quote(quote))
    print("   理由:", decision["reason"])
    print()


01. issue_9bebea1e5c499a9efb51036b | train:671, train:50554
   建议动作: quarantine_training_conflict_cluster
   建议隔离: train:671, train:50554
   冲突方面: dish_look, dish_portion, dish_taste, environment_decoration, others_overall_experience, price_cost_effective, price_level, service_waiters_attitude
   证据: "量和传统的比也确实袖珍了些"
   证据: "所以食物方面我一颗星都打不出。"
   证据: "抓范实在算不上是性价比高的"
   理由: 两条非空白字符完全一致，仅段落缩进不同。原文明确表达分量少、味道差和性价比不足，但两个成员在8个方面有分歧。671的多项标签较符合原文，其性价比中性标签仍可疑；不能认定任一成员整行20标签都正确。建议整簇隔离训练，待统一口径复核全部方面后再保留一个代表；本次仅完成复核建议，未修复数据。

02. issue_2d86e8fb5cc7e78ad99bfc3a | train:4598, train:73456
   建议动作: quarantine_training_conflict_cluster
   建议隔离: train:4598, train:73456
   冲突方面: price_cost_effective, price_discount
   证据: "两人消费很划算。"
   证据: "量大实惠"
   证据: "套餐很划算"
   理由: 两条非空白字符完全一致，仅连续空格数量不同。性价比有明确正向证据，4598的该项未提及标签可疑；优惠标签正向与中性的分歧仍依赖团购提及和优惠评价的统一边界。全文有拼接式和混合评价，不能只修两项便认定全行可靠。建议整簇隔离，待统一口径复核全部20方面后保留一个代表；未执行修复。

03. issue_6eea30ef8ac8eaf74d9db2ed | train:8429, train:79897
   建议动作: quarantine_training_conflict_clu

## 13 条异常字符告警

只列出已复核的代码点、动作及短引文。普通换行、表情和情绪文本保留；替换字符或大量残损片段不猜测补字。对固定验证集记录的字符处理只建议在版本化副本或一致的模型预处理中执行，raw 不变。


In [5]:
warning_items = [d for d in decisions if d["original_severity"] == "WARN"]
for index, decision in enumerate(warning_items, 1):
    print(f"{index:02d}. {decision['issue_id']} | {references(decision['affected_records'])}")
    print("   字符:", json.dumps(decision["characters"], ensure_ascii=False, sort_keys=True))
    print("   建议动作:", decision["action"]["type"])
    for quote in decision["evidence_quotes"]:
        print("   证据:", show_quote(quote))
    print("   理由:", decision["reason"])
    print()


01. issue_3479d65e65b68308a1b0a4a2 | train:64
   字符: {"U+0005": 35, "U+0006": 34, "U+0007": 42}
   建议动作: quarantine_corrupted_training_record
   证据: "红汤锅\u0007\u0006\u0006\u0006\u0007\u0006\u0005\u0007\u0007\u0007\u0006bbb"
   证据: "需求bbb\u0007\u0005\u0007\u0005b"
   证据: "特色火锅味b\u0005\u0005\u0007\u0005\u0007"
   理由: 293个字符中有111个异常控制符（37.88%），另有39个b，与控制符反复交织。单删控制符仍残留大段b串，无法仅靠无损字符删除恢复干净文本；首版隔离整条，保留原始记录及标签，不对全局英文b建立删除规则。

02. issue_4b024e08d70f79bb68faef42 | train:7649
   字符: {"U+007F": 5}
   建议动作: remove_reviewed_control_characters
   证据: "空气中弥漫着慵懒的蓝调。\u007f\n站在窗前"
   证据: "独特记号。\n\u007f\n\n金枪鱼刺身"
   证据: "\u007f\n德国猪肘"
   理由: 5个U+007F位于句尾或段落、标题边界，不承担可读词义。仅删除已定位控制符即可保留正文和原换行，标签不变。

03. issue_d967e5437f45de6cc930b0d7 | train:12065
   字符: {"U+001C": 2, "U+001E": 1}
   建议动作: remove_reviewed_control_characters
   证据: "于是来到这家菜馆\u001e。"
   证据: "薯粉和鸡蛋做的，\u001c蛋液"
   证据: "就是有点点\u001c油腻"
   理由: 3个控制符分别位于句末、分句起点及“点点油腻”之间，删除后中文语句自然连续。仅删除控制符，不插入会拆开词组的空格，不改标签。

04. issue_4315b17d780922c655b337ab | train

## 边界与后续条件

- 此处完成的是复核与只读预演，**没有执行动作**，原始问题不能标记为已修复。
- 固定验证集中两条跨集合记录的标签仍有争议；隔离训练重叠只消除泄漏，不能消除参考标签噪声。
- 如采纳建议，应生成有版本的 processed 副本，记录隔离与字符修改日志，并重新执行校验。
- 全量预演没有新增空文本、已定义的异常字符或归一化重复，不保证消除了所有语义相似样本或标注错误。
- CSV 没有评论时间戳，因此不作时间趋势结论。
